# 🎙️ Matraca Studio — Dublador & Clonador de Voz com IA
### Envie seu vídeo MP4 ou áudio, clone sua própria voz e exporte o conteúdo dublado exatamente no mesmo tempo do original!

Este notebook executa um pipeline completo e profissional de localização de vídeo e áudio:
1. **Entrada Universal:** Suporte para arquivos de vídeo (`.mp4`, `.mov`, `.mkv`) ou áudio (`.wav`, `.mp3`, `.m4a`) e microfone.
2. **Reconhecimento de Fala (Whisper):** Transcreve com alta precisão e detecta automaticamente o idioma original (executado apenas uma vez para otimização máxima).
3. **Seleção Multi-Idioma (Checkboxes):** Escolha um ou vários idiomas de destino (**Inglês, Espanhol, Francês, Alemão, Chinês, Árabe**, etc.) para dublagem em lote sequencial.
4. **Tradução Automática Segmentada:** Tradução robusta sem limite de caracteres por final de frase natural.
5. **Clonagem de Voz com IA (OmniVoice):** Recria a sua voz falando em cada idioma selecionado mantendo timbre, entonação e características vocais únicas.
6. **Sincronização Temporal com o Vídeo Original:** Aplica time-stretching inteligente com preservação total de tom (*pitch-preserved atempo* via FFmpeg), garantindo que cada áudio dublado case perfeitamente com a duração do vídeo.
7. **Download Individual de Mídia:** Gera e disponibiliza para **download individual** cada áudio WAV sincronizado e vídeo MP4 dublado gerado para cada idioma selecionado.

> ⚠️ **Requisito Obrigatório (GPU T4):**
> Vá no menu superior do Colab em **Ambiente de Execução (Runtime)** ➔ **Alterar tipo de ambiente de execução (Change runtime type)** ➔ Selecione **T4 GPU**.


In [ ]:
# @title Passo 1: Instalar Dependências e FFmpeg
# @markdown Instala OmniVoice, Whisper, Gradio, Deep-Translator e ferramentas de mídia.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg
!pip install -q omnivoice gradio openai-whisper deep-translator
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
print('✅ Dependências e ferramentas de mídia instaladas com sucesso!')

In [ ]:
# @title Passo 2: Carregar os Modelos de IA na GPU (T4 / A100 / L4)
# @markdown Baixa e carrega o Whisper e o OmniVoice na memória de vídeo da GPU.

import os
import torch
import torchaudio
import whisper
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'🖥️ Dispositivo em uso: {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU Detectada: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ GPU não detectada! Por favor, ative a T4 GPU no menu do Colab (Runtime > Change runtime type).')

print('\n⏳ Carregando modelo Whisper (base) para transcrição de áudio...')
whisper_model = whisper.load_model('base', device=device)
print('✅ Whisper pronto!')

print('\n⏳ Carregando OmniVoice da k2-fsa (primeira vez baixa os pesos do modelo)...')
omnivoice_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ OmniVoice carregado e pronto para clonar sua voz!')

In [ ]:
# @title Passo 3: Motor de Processamento, Sincronização Temporal e Remuxing de Vídeo
# @markdown Funções auxiliares para extração de áudio, medição de duração, ajuste atempo e geração do MP4 final.

import subprocess
import json
import tempfile
import os
import re
import gc
import time
import torch
import torchaudio
from deep_translator import GoogleTranslator

def get_media_info(file_path):
    """Retorna a duração em segundos e se o arquivo contém faixa de vídeo."""
    if not file_path or not os.path.exists(file_path):
        return 0.0, False
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration:stream=codec_type',
        '-of', 'json', file_path
    ]
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(res.stdout)
        duration = float(data.get('format', {}).get('duration', 0.0))
        streams = data.get('streams', [])
        has_video = any(s.get('codec_type') == 'video' for s in streams)
        return duration, has_video
    except Exception as e:
        print(f'Erro ao inspecionar mídia com ffprobe: {e}')
        return 0.0, False

def extract_audio_to_wav(media_path, output_wav):
    """Converte qualquer mídia (vídeo ou áudio) em WAV 24kHz mono puro para o OmniVoice."""
    cmd = [
        'ffmpeg', '-y', '-i', media_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def extract_audio_slice(input_wav, start_sec, end_sec, output_slice_wav):
    """
    Extrai uma fatia pura de áudio (ex: 5 a 10s) sem filtros artificiais
    para preservar 100% do timbre e formantes naturais da voz original.
    """
    duration = max(0.5, end_sec - start_sec)
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_sec:.3f}',
        '-t', f'{duration:.3f}',
        '-i', input_wav,
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_slice_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def translate_text_robust(text, target_code, max_chunk=2500):
    """
    Traduz textos de qualquer tamanho sem estourar o limite de 5000 caracteres.
    Suporta pt-BR, es, en, etc., com fallback inteligente por pontuação natural.
    """
    if not text or not text.strip():
        return ''
    clean_text = text.strip()
    api_target = 'pt' if target_code.lower() in ('pt-br', 'pt_br') else target_code
    if len(clean_text) < max_chunk:
        try:
            return GoogleTranslator(source='auto', target=api_target).translate(clean_text)
        except Exception:
            pass

    # Divide por pontuação natural de fim de frase
    sentences = re.split(r'(?<=[.!?;\n])\s+', clean_text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    translator = GoogleTranslator(source='auto', target=api_target)
    translated_parts = []
    for chunk in chunks:
        chunk = chunk.strip()
        if chunk:
            part = None
            try:
                part = translator.translate(chunk)
                time.sleep(0.1)
            except Exception:
                try:
                    from deep_translator import MyMemoryTranslator
                    part = MyMemoryTranslator(source='auto', target='pt-BR' if api_target == 'pt' else api_target).translate(chunk)
                except Exception:
                    part = chunk
            if part:
                translated_parts.append(part)
    return ' '.join(translated_parts)

def build_atempo_filter(speed_factor):
    """Gera encadeamento de filtros atempo no FFmpeg (cada filtro suporta entre 0.5 e 2.0)."""
    speed = speed_factor
    filters = []
    while speed > 2.0:
        filters.append('atempo=2.0')
        speed /= 2.0
    while speed < 0.5:
        filters.append('atempo=0.5')
        speed /= 0.5
    filters.append(f'atempo={speed:.5f}')
    return ','.join(filters)

def time_sync_audio(synth_wav_path, target_duration, output_synced_wav):
    """
    Ajusta a velocidade do áudio gerado para casar exatamente com a duração do original,
    preservando o tom e timbre da voz clonada (pitch-preserved time stretch).
    """
    synth_duration, _ = get_media_info(synth_wav_path)
    if synth_duration <= 0 or target_duration <= 0:
        cmd = ['ffmpeg', '-y', '-i', synth_wav_path, '-c', 'copy', output_synced_wav]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return 1.0, synth_duration

    speed_factor = synth_duration / target_duration

    # Se a diferença de duração for mínima (< 1.5%), mantém ritmo natural e ajusta preenchimento
    if 0.985 <= speed_factor <= 1.015:
        filter_chain = f'apad=whole_dur={target_duration:.4f}'
    else:
        tempo_filter = build_atempo_filter(speed_factor)
        filter_chain = f'{tempo_filter},apad=whole_dur={target_duration:.4f}'

    cmd = [
        'ffmpeg', '-y', '-i', synth_wav_path,
        '-filter:a', filter_chain,
        '-t', f'{target_duration:.4f}',
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_synced_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return speed_factor, synth_duration

def remux_video_with_audio(original_video_path, new_audio_path, output_video_path):
    """
    Substitui a faixa de áudio do vídeo original pelo áudio dublado e sincronizado.
    Utiliza stream copy (-c:v copy) para renderização instantânea sem perda de qualidade visual.
    """
    cmd = [
        'ffmpeg', '-y',
        '-i', original_video_path,
        '-i', new_audio_path,
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-b:a', '192k',
        '-shortest',
        output_video_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def generate_voice_cloning_chunked(model, text, ref_audio, ref_text, num_steps=32, speed=1.0, max_chunk_chars=300, progress_callback=None):
    """
    Gera fala com OmniVoice dividindo textos longos em frases naturais para máxima fidelidade,
    evitando estouro de VRAM na GPU e garantindo entonação e timbre constantes.
    """
    if len(text) <= max_chunk_chars:
        if progress_callback:
            progress_callback(0, 1, text)
        out = model.generate(text=text, ref_audio=ref_audio, ref_text=ref_text, num_step=int(num_steps), speed=float(speed))
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        return t.unsqueeze(0) if t.dim() == 1 else t

    # Divide em frases naturais preservando pontuação
    sentences = re.split(r'(?<=[.!?;\n])\s+', text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk_chars:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    print(f'🎙️ Sintetizando áudio em {len(chunks)} blocos naturais de fala...')
    tensors = []
    silence = torch.zeros((1, int(24000 * 0.15)))
    for idx, c in enumerate(chunks):
        print(f'  - Gerando bloco {idx+1}/{len(chunks)}: {c[:45]}...')
        if progress_callback:
            progress_callback(idx, len(chunks), c)
        out = model.generate(text=c, ref_audio=ref_audio, ref_text=ref_text, num_step=int(num_steps), speed=float(speed))
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        if t.dim() == 1:
            t = t.unsqueeze(0)
        tensors.append(t)
        tensors.append(silence)

    if tensors:
        tensors.pop()
        return torch.cat(tensors, dim=-1)
    return torch.zeros((1, 24000))

print('✅ Motor de processamento, tradução robusta e sincronização carregado!')


In [ ]:
# @title Passo 4: Iniciar a Interface de Dublagem Sincronizada (Gradio)
# @markdown Clique no botão 'Play' e acesse o link público 'Running on public URL: https://...gradio.live'

import os
import re
import shutil
import tempfile
import gc
import torch
import torchaudio
import gradio as gr

# Idiomas suportados com destaque para Português Brasileiro e idiomas globais
LANGUAGES = {
    '🇧🇷 Português Brasileiro (pt-BR)': 'pt-BR',
    '🇺🇸 Inglês (English)': 'en',
    '🇪🇸 Espanhol (Español)': 'es',
    '🇫🇷 Francês (Français)': 'fr',
    '🇩🇪 Alemão (Deutsch)': 'de',
    '🇨🇳 Chinês Simplificado (中文)': 'zh-CN',
    '🇸🇦 Árabe (العربية)': 'ar',
    '🇮🇹 Italiano (Italiano)': 'it',
    '🇯🇵 Japonês (日本語)': 'ja',
    '🇷🇺 Russo (Русский)': 'ru'
}

def resolve_lang_code(lang_label):
    """Garante a correspondência perfeita entre a escolha visual e o código de tradução."""
    if lang_label in LANGUAGES:
        return LANGUAGES[lang_label]
    l_lower = lang_label.lower()
    if 'espanh' in l_lower or 'es' in l_lower:
        return 'es'
    if 'ingl' in l_lower or 'en' in l_lower:
        return 'en'
    if 'portug' in l_lower or 'pt' in l_lower:
        return 'pt-BR'
    if 'franc' in l_lower or 'fr' in l_lower:
        return 'fr'
    if 'alem' in l_lower or 'de' in l_lower:
        return 'de'
    if 'chin' in l_lower or 'zh' in l_lower:
        return 'zh-CN'
    if 'arab' in l_lower or 'ar' in l_lower:
        return 'ar'
    if 'ital' in l_lower or 'it' in l_lower:
        return 'it'
    if 'japon' in l_lower or 'ja' in l_lower:
        return 'ja'
    if 'russ' in l_lower or 'ru' in l_lower:
        return 'ru'
    return 'en'

def transcribe_only(media_file, progress=gr.Progress()):
    """Transcreve o áudio original com Whisper para permitir conferência e ajuste prévio."""
    if not media_file:
        return '', '❌ Por favor, envie um arquivo de vídeo ou áudio antes de transcrever.'
    try:
        if progress is not None:
            progress(0.15, desc='Extraindo faixa de áudio...')

        orig_dur, _ = get_media_info(media_file)
        temp_wav = tempfile.NamedTemporaryFile(suffix='_transcribe.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_wav)

        if progress is not None:
            progress(0.50, desc='Transcrevendo fala com Whisper...')
        asr_res = whisper_model.transcribe(temp_wav)
        text = asr_res.get('text', '').strip()
        lang = asr_res.get('language', 'desconhecido')

        if progress is not None:
            progress(1.0, desc='Transcrição concluída!')

        status_msg = f"""✅ **Transcrição concluída com sucesso!**
- ⏱️ **Duração da Mídia:** `{orig_dur:.2f}s` | 🌐 **Idioma detectado:** `{lang.upper()}`
> 💡 *Você pode ler e editar qualquer palavra no campo abaixo. Em seguida, selecione os idiomas e clique em **'2. Dublar e Sincronizar'**.*"""
        return text, status_msg
    except Exception as e:
        return '', f'❌ **Erro ao transcrever áudio:** `{str(e)}`'

def process_dubbing(media_file, edited_transcription, target_lang_labels, sync_duration_opt, num_steps, user_speed, progress=gr.Progress()):
    if not media_file:
        return (
            [],
            '❌ **Erro:** Por favor, envie um arquivo de vídeo (.mp4, .mov, etc.) ou de áudio (.wav, .mp3, etc.).',
            '', ''
        )

    if not target_lang_labels or len(target_lang_labels) == 0:
        return (
            [],
            '❌ **Erro:** Por favor, marque pelo menos um idioma de destino nas caixas de seleção.',
            '', ''
        )

    try:
        if progress is not None:
            progress(0.05, desc='Analisando arquivo de mídia...')

        orig_duration, has_video = get_media_info(media_file)

        # 1. Extrair áudio completo para WAV a 24kHz
        temp_full_wav = tempfile.NamedTemporaryFile(suffix='_full.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_full_wav)

        # 2. Executar Whisper para obter os segmentos e timestamps exatos da voz original
        if progress is not None:
            progress(0.12, desc='Analisando voz original com Whisper...')
        print('🎙️ Identificando segmentos vocais com Whisper...')
        asr_result = whisper_model.transcribe(temp_full_wav)
        whisper_text = asr_result.get('text', '').strip()
        detected_lang = asr_result.get('language', 'desconhecido')
        segments = asr_result.get('segments', [])

        # Transcrição a ser traduzida: se o usuário revisou/editou, respeita o texto do usuário!
        original_text = (edited_transcription or '').strip()
        if not original_text:
            original_text = whisper_text

        if not original_text:
            return (
                [],
                '❌ **Erro:** Não foi possível reconhecer fala audível no arquivo enviado.',
                '', ''
            )

        # 3. Selecionar amostra vocal ideal de referência (3 a 10 segundos) baseada nos segmentos do Whisper
        ref_slice_wav = tempfile.NamedTemporaryFile(suffix='_ref_slice.wav', delete=False).name
        ref_slice_start = 0.0
        ref_slice_end = min(orig_duration, 10.0)
        ref_slice_text = original_text[:120]

        if segments:
            seg_texts = []
            ref_slice_start = segments[0]['start']
            ref_slice_end = segments[0]['end']
            for s in segments:
                seg_texts.append(s['text'].strip())
                ref_slice_end = s['end']
                if (ref_slice_end - ref_slice_start) >= 5.0:
                    break
            ref_slice_text = ' '.join(seg_texts)

        # Extração pura da fatia vocal (sem filtros artificiais que distorçam o timbre)
        extract_audio_slice(temp_full_wav, ref_slice_start, ref_slice_end, ref_slice_wav)

        # Diretório dedicado para os arquivos gerados nesta execução em lote
        output_dir = tempfile.mkdtemp(prefix='matraca_batch_')
        orig_base_name = os.path.splitext(os.path.basename(media_file))[0]
        clean_base_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', orig_base_name)

        generated_files = []
        translations_summary = []
        results_info = []

        total_langs = len(target_lang_labels)
        print(f'🚀 Iniciando lote de dublagem para {total_langs} idioma(s): {target_lang_labels}')

        # 4. Processamento sequencial otimizado de cada idioma selecionado
        for idx, lang_label in enumerate(target_lang_labels):
            lang_code = resolve_lang_code(lang_label)
            clean_code = lang_code.replace('-', '_')
            step_base = 0.20 + (0.75 * idx / total_langs)
            lang_span = 0.75 / total_langs

            print(f'\n--- [{idx+1}/{total_langs}] Processando idioma: {lang_label} ({lang_code}) ---')
            if progress is not None:
                progress(step_base, desc=f'[{idx+1}/{total_langs}] Traduzindo para {lang_label}...')

            # Tradução robusta no idioma selecionado
            print(f'🌍 Traduzindo do {detected_lang} para {lang_label} ({lang_code})...')
            translated_text = translate_text_robust(original_text, lang_code)
            translations_summary.append(f'### {lang_label}\n{translated_text}\n')

            # Síntese vocal clonada com atualização bloco a bloco
            cloning_start = step_base + (lang_span * 0.15)
            cloning_span = lang_span * 0.60

            def chunk_progress(chunk_idx, num_chunks, chunk_txt):
                if progress is not None:
                    p = cloning_start + (cloning_span * (chunk_idx + 1) / max(1, num_chunks))
                    progress(p, desc=f'[{idx+1}/{total_langs}] {lang_label}: Bloco {chunk_idx+1}/{num_chunks}...')

            if progress is not None:
                progress(cloning_start, desc=f'[{idx+1}/{total_langs}] Clonando voz em {lang_label}...')

            print(f'🧬 Sintetizando fala clonada com OmniVoice ({lang_label})...')
            audio_tensor = generate_voice_cloning_chunked(
                model=omnivoice_model,
                text=translated_text,
                ref_audio=ref_slice_wav,
                ref_text=ref_slice_text,
                num_steps=int(num_steps),
                speed=float(user_speed),
                progress_callback=chunk_progress
            )

            temp_synth_wav = tempfile.NamedTemporaryFile(suffix='_synth.wav', delete=False).name
            torchaudio.save(temp_synth_wav, audio_tensor.cpu(), 24000)

            # Sincronização temporal milimétrica com FFmpeg
            if progress is not None:
                progress(step_base + (lang_span * 0.80), desc=f'[{idx+1}/{total_langs}] Sincronizando tempo do áudio...')

            final_audio_path = os.path.join(output_dir, f'{clean_base_name}_{clean_code}_audio.wav')
            speed_factor = 1.0
            synth_duration = audio_tensor.shape[-1] / 24000.0

            if sync_duration_opt and orig_duration > 0:
                print(f'⏱️ Sincronizando duração: {synth_duration:.2f}s ➔ {orig_duration:.2f}s...')
                speed_factor, synth_duration = time_sync_audio(temp_synth_wav, orig_duration, final_audio_path)
                final_duration = orig_duration
            else:
                shutil.copyfile(temp_synth_wav, final_audio_path)
                final_duration = synth_duration

            generated_files.append(final_audio_path)

            # Remuxing do vídeo MP4 (se o arquivo original tiver vídeo)
            final_video_path = None
            if has_video:
                if progress is not None:
                    progress(step_base + (lang_span * 0.92), desc=f'[{idx+1}/{total_langs}] Renderizando vídeo MP4 final...')
                print(f'🎬 Integrando áudio dublado ao vídeo original ({lang_label})...')
                final_video_path = os.path.join(output_dir, f'{clean_base_name}_{clean_code}_dublado.mp4')
                remux_video_with_audio(media_file, final_audio_path, final_video_path)
                generated_files.append(final_video_path)

            results_info.append({
                'label': lang_label,
                'code': lang_code,
                'synth_dur': synth_duration,
                'final_dur': final_duration,
                'speed_factor': speed_factor,
                'audio_file': os.path.basename(final_audio_path),
                'video_file': os.path.basename(final_video_path) if final_video_path else None
            })

            # Liberação obrigatória de VRAM após cada idioma
            print(f'🧹 Liberando memória GPU após [{lang_label}]...')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        if progress is not None:
            progress(1.0, desc='Processamento em lote concluído com sucesso!')

        # Construir tabela resumo de métricas em Markdown
        rows_md = []
        for r in results_info:
            vid_col = f"`{r['video_file']}`" if r['video_file'] else 'N/A (Entrada de Áudio)'
            rows_md.append(f"| {r['label']} | `{r['audio_file']}` | {vid_col} | `{r['speed_factor']:.3f}x` | `{r['final_dur']:.2f}s` |")

        table_body = '\n'.join(rows_md)
        status_md = f"""### ✅ Dublagem em Lote Concluída com Sucesso! ({len(results_info)} idioma(s) gerado(s))
- 🌐 **Idioma Original Detectado:** `{detected_lang.upper()}`
- ⏱️ **Duração do Original:** `{orig_duration:.2f}s`
- 📥 **Downloads Individuais:** {len(generated_files)} arquivo(s) disponíveis para download individual abaixo.

| Idioma | Áudio WAV | Vídeo MP4 Dublado | Ajuste (atempo) | Duração Final |
| :--- | :--- | :--- | :--- | :--- |
{table_body}
"""

        all_translations = '\n\n---\n\n'.join(translations_summary)
        return generated_files, status_md, original_text, all_translations

    except Exception as e:
        import traceback
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return (
            [],
            f'❌ **Erro durante a execução:** `{str(e)}`',
            edited_transcription or '', ''
        )

def process_free_tts(custom_text, ref_audio, num_steps, speed):
    if not custom_text.strip():
        return None, '❌ Digite um texto para sintetizar.'
    if not ref_audio:
        return None, '❌ Envie uma amostra de áudio com a voz a ser clonada.'
    try:
        audio_tensor = generate_voice_cloning_chunked(
            model=omnivoice_model,
            text=custom_text,
            ref_audio=ref_audio,
            ref_text='',
            num_steps=int(num_steps),
            speed=float(speed)
        )
        tmp_file = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        torchaudio.save(tmp_file.name, audio_tensor.cpu(), 24000)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return tmp_file.name, '✅ Fala sintetizada com sucesso com a sua voz!'
    except Exception as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return None, f'❌ Erro: {str(e)}'

# Interface Gráfica Moderna
with gr.Blocks(title='Matraca Studio — Dublador & Clonador de Voz com IA', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue')) as demo:
    gr.HTML("""
    <div style='text-align: center; margin-bottom: 18px;'>
        <h1 style='font-size: clamp(1.2rem, 3.2vw, 2.0rem); font-weight: 700; white-space: nowrap; margin-bottom: 6px;'>🎙️ Matraca Studio — Dublador & Clonador de Voz com IA</h1>
        <p style='color: #555; font-size: 1.1em;'>
            Clone sua voz e duble qualquer vídeo MP4 ou áudio para <b>múltiplos idiomas</b>
            mantendo <b>exatamente o mesmo tempo de duração</b> do vídeo original!
        </p>
    </div>
    """)

    with gr.Tabs():
        # --- ABA 1: DUBLAGEM SINCRONIZADA ---
        with gr.TabItem('🎬 Dublagem Sincronizada (Vídeo ou Áudio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_media = gr.File(
                        label='1. Envie seu Vídeo (MP4, MOV, MKV) ou Áudio (WAV, MP3, M4A)',
                        file_count='single',
                        type='filepath'
                    )
                    btn_transcribe = gr.Button('📝 1. Transcrever e Analisar Áudio Original', variant='secondary', size='md')
                    status_transcribe = gr.Markdown('')

                    txt_orig = gr.Textbox(
                        label='Transcrição do Áudio Original (Revise e edite as palavras se desejar):',
                        placeholder='Clique em "1. Transcrever e Analisar Áudio Original" acima para transcrever, ou digite/cole o texto aqui...',
                        lines=4,
                        interactive=True
                    )

                    target_langs = gr.CheckboxGroup(
                        choices=list(LANGUAGES.keys()),
                        value=['🇺🇸 Inglês (English)', '🇪🇸 Espanhol (Español)'],
                        label='2. Idiomas de Destino da Dublagem (Selecione um ou vários)',
                        info='Marque os idiomas desejados. Cada um será processado sequencialmente de forma otimizada.'
                    )
                    with gr.Row():
                        btn_select_all = gr.Button('☑️ Selecionar Todos', size='sm')
                        btn_clear_all = gr.Button('⬜ Limpar Seleção', size='sm')

                    sync_checkbox = gr.Checkbox(
                        value=True,
                        label='⏱️ Sincronizar Duração com o Original (Garante mesmo tempo do vídeo)',
                        info='Ajusta a velocidade da fala mantendo o tom natural e timbre da sua voz clonada.'
                    )
                    with gr.Accordion('⚙️ Configurações Avançadas de IA', open=False):
                        steps_slider = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão (Diffusion Steps)')
                        speed_slider = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label='Velocidade Base da Fala')

                    btn_dub = gr.Button('✨ 2. Dublar e Sincronizar Vídeo/Áudio', variant='primary', size='lg')

                with gr.Column(scale=1):
                    status_label = gr.Markdown('')
                    output_files = gr.File(
                        label='📥 Download Individual dos Arquivos Gerados (Áudios WAV e Vídeos MP4)',
                        file_count='multiple',
                        type='filepath',
                        interactive=False
                    )
                    with gr.Accordion('📜 Traduções Geradas por Idioma', open=True):
                        txt_trans = gr.Textbox(label='Traduções Geradas por Idioma', lines=8, interactive=False)

            btn_select_all.click(fn=lambda: list(LANGUAGES.keys()), outputs=[target_langs])
            btn_clear_all.click(fn=lambda: [], outputs=[target_langs])

            btn_transcribe.click(
                fn=transcribe_only,
                inputs=[input_media],
                outputs=[txt_orig, status_transcribe]
            )

            btn_dub.click(
                fn=process_dubbing,
                inputs=[input_media, txt_orig, target_langs, sync_checkbox, steps_slider, speed_slider],
                outputs=[output_files, status_label, txt_orig, txt_trans]
            )

        # --- ABA 2: CLONAGEM LIVRE ---
        with gr.TabItem('✍️ Clonagem Livre (Digitar Texto Personalizado)'):
            gr.Markdown('Envie uma amostra de áudio com a sua voz e digite qualquer texto em qualquer idioma para sintetizar diretamente:')
            with gr.Row():
                with gr.Column(scale=1):
                    ref_audio_free = gr.Audio(label='Áudio de Referência (sua voz)', type='filepath')
                    custom_text = gr.Textbox(
                        label='Texto a ser falado',
                        placeholder='Ex: Hello everyone! Today we are introducing our new AI-powered dubbing technology.',
                        lines=4
                    )
                    with gr.Row():
                        steps_free = gr.Slider(minimum=16, maximum=64, value=32, step=8, label='Passos de Difusão')
                        speed_free = gr.Slider(minimum=0.5, maximum=1.5, value=1.0, step=0.1, label='Velocidade')
                    btn_free = gr.Button('Gerar Áudio com Minha Voz', variant='primary', size='lg')
                    status_free = gr.Markdown('')
                with gr.Column(scale=1):
                    audio_free_out = gr.Audio(label='Áudio Sintetizado', type='filepath', interactive=False)

            btn_free.click(
                fn=process_free_tts,
                inputs=[custom_text, ref_audio_free, steps_free, speed_free],
                outputs=[audio_free_out, status_free]
            )

demo.launch(share=True, debug=True)
